# 12 — Information Theory for Machine Learning Practice

This notebook practices information content, entropy, cross-entropy, KL divergence, mutual information, information gain, and perplexity.

In [ ]:
import numpy as np

## 1. Information Content

$$
I(p)=-\log_2(p)
$$

In [ ]:
def information_content(p, base=2):
    return -np.log(p) / np.log(base)

for p in [1.0, 0.5, 0.25, 0.125]:
    print(p, information_content(p))

## 2. Entropy

$$
H(X)=-\sum_x p(x)\log_2 p(x)
$$

In [ ]:
def entropy(probs, base=2):
    probs = np.array(probs, dtype=float)
    probs = probs[probs > 0]
    logs = np.log(probs) / np.log(base)
    return -np.sum(probs * logs)

certain = np.array([1.0, 0.0, 0.0, 0.0])
skewed = np.array([0.70, 0.15, 0.10, 0.05])
uniform = np.array([0.25, 0.25, 0.25, 0.25])

entropy(certain), entropy(skewed), entropy(uniform)

## 3. Cross-Entropy and KL Divergence

$$
H(P,Q)=-\sum_x P(x)\log Q(x)
$$

$$
D_{KL}(P\|Q)=\sum_xP(x)\log\frac{P(x)}{Q(x)}
$$

$$
H(P,Q)=H(P)+D_{KL}(P\|Q)
$$

In [ ]:
def cross_entropy(p_true, q_model, base=2, eps=1e-15):
    p_true = np.array(p_true, dtype=float)
    q_model = np.array(q_model, dtype=float)
    q_model = np.clip(q_model, eps, 1)
    logs = np.log(q_model) / np.log(base)
    return -np.sum(p_true * logs)


def kl_divergence(p_true, q_model, base=2, eps=1e-15):
    p_true = np.array(p_true, dtype=float)
    q_model = np.array(q_model, dtype=float)
    mask = p_true > 0
    p = p_true[mask]
    q = np.clip(q_model[mask], eps, 1)
    logs = np.log(p / q) / np.log(base)
    return np.sum(p * logs)

P = np.array([0.55, 0.25, 0.15, 0.05])
Q = np.array([0.50, 0.28, 0.15, 0.07])

entropy(P), cross_entropy(P, Q), kl_divergence(P, Q), entropy(P) + kl_divergence(P, Q)

## 4. Mutual Information

$$
I(X;Y)=\sum_{x,y}P(x,y)\log\frac{P(x,y)}{P(x)P(y)}
$$

In [ ]:
def mutual_information(joint, base=2, eps=1e-15):
    joint = np.array(joint, dtype=float)
    px = joint.sum(axis=1, keepdims=True)
    py = joint.sum(axis=0, keepdims=True)
    expected = px @ py
    mask = joint > 0
    ratio = joint[mask] / np.clip(expected[mask], eps, None)
    logs = np.log(ratio) / np.log(base)
    return np.sum(joint[mask] * logs)

joint_independent = np.array([[0.25, 0.25], [0.25, 0.25]])
joint_dependent = np.array([[0.45, 0.05], [0.05, 0.45]])

mutual_information(joint_independent), mutual_information(joint_dependent)

## 5. Information Gain

$$
IG(Y,X)=H(Y)-H(Y|X)
$$

In [ ]:
def information_gain(parent_labels, left_labels, right_labels):
    def label_entropy(labels):
        values, counts = np.unique(labels, return_counts=True)
        probs = counts / counts.sum()
        return entropy(probs)

    parent_entropy = label_entropy(parent_labels)
    n = len(parent_labels)
    weighted_child_entropy = (
        len(left_labels) / n * label_entropy(left_labels)
        + len(right_labels) / n * label_entropy(right_labels)
    )
    return parent_entropy - weighted_child_entropy

parent = np.array([0, 0, 0, 0, 1, 1, 1, 1])
left = np.array([0, 0, 0, 1])
right = np.array([0, 1, 1, 1])

information_gain(parent, left, right)

## 6. Perplexity

If cross-entropy is in bits:

$$
\mathrm{perplexity}=2^H
$$

In [ ]:
def perplexity_from_bits(cross_entropy_bits):
    return 2 ** cross_entropy_bits

perplexity_from_bits(2.5)

## Reflection

Information theory connects uncertainty, compression, classification loss, decision trees, language modeling, and representation learning.